# Full M1 candidate — Qwen3-1.7B, F30/R90

This notebook **actually trains a new full M1 candidate** from the pinned public base. It does not continue the tiny adapter. Reviewed candidate v1.1 supplies120 F/R units,840 original training questions,3 training-only wording variants, and both answer-only/canonical-chain targets:5040 rows. N_rel/N_reason and all final-test prompts remain excluded from training/model final evaluation.

Fixed budget:1200 attempted updates (10x tiny data/attempts; about3.81 dataset passes), same LoRA/batch/LR/precision/loss. Back up adapter+optimizer+tokenizer at attempts300/600/900/1200 to the declared private HF repository and verify immutable hash round-trips. Only the1200-attempt endpoint is eligible; intermediate checkpoints are backups, never selected by development score. Checkpoints do not implement exact mid-run resume.

First prepare tokens and run a disposable longest-batch memory probe in a separate process. Actual training then starts fresh, checks32 exact fact replay controls and10 measured utility answers, and reuses the already-verified1680-row M0 F/R baseline. A replay failure stops before adapter creation. No silent batch/precision/mode fallback or budget extension.

Evaluate the endpoint on840 development questions in each mode and10 thinking arithmetic probes, then repeat from a fresh-base reload:3380 endpoint/reload outputs. Primary research setting is thinking; non-thinking remains a same-checkpoint access check. Require all original tiny thresholds plus >=90% per one-hop task and >=75% per multi-hop task in F and R separately, in both modes; zero post format/cap/boundary failures, >=8/10 measurable base utility, <=1/10 utility drop, all1200 updates, adapter change and exact save/reload outputs/tokens. The ten arithmetic probes are reused diagnostics, not a broad independent utility benchmark. Chains/composite answers are supervised, not proof of unseen-composition reasoning.

Publish source/config/reviewed corrected-baseline evidence first. Use the same Kaggle T4 environment and HF_TOKEN secret. Run sequentially, preserve failed outcomes, finish private backup and download the evidence ZIP before stopping the runtime. Full GPU fit/learning/backup remain untested until this run; passing gates still needs independent review before canonical M1 promotion. M2/unlearning is a later stage.


In [1]:
import os, subprocess, sys, tempfile
from pathlib import Path

remote_root = Path('/kaggle/working')
assert remote_root.is_dir(), 'Connect to the Kaggle GPU kernel.'
repo_dir = Path(tempfile.mkdtemp(prefix='full-m1-source-', dir=remote_root)) / 'repo'
subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/DevulapalliTharun/persistent-llm-unlearning.git', str(repo_dir)], check=True)
os.chdir(repo_dir)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
assert Path('src/training/full_m1.py').is_file(), 'Publish the prepared full-M1 source/config/results first.'
assert Path('experiments/runs/m0_qwen17_corrected_20261107/data_review.json').is_file()
config_path = 'configs/training/m1_full_qwen17_v1.json'


Cloning into '/kaggle/working/full-m1-source-k0g6mi8k/repo'...


Repository commit: 35b961e914a57a0fc09cd4c9d02e634670778d73


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', 'requirements/m1-tiny.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
subprocess.run([sys.executable, '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))'], check=True)


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
2.11.0+cu130 5.20.2 0.21.2 Tesla T4


CompletedProcess(args=['/usr/bin/python3', '-c', 'import torch, transformers, peft; from peft.import_utils import is_torchao_available; assert not is_torchao_available(); assert torch.cuda.is_available(); print(torch.__version__, transformers.__version__, peft.__version__, torch.cuda.get_device_name(0))'], returncode=0)

In [3]:
import json
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

hf_token = UserSecretsClient().get_secret('HF_TOKEN')
config = json.loads(Path(config_path).read_text())
api = HfApi(token=hf_token)
assert api.repo_info(config['persistence']['repo_id'], repo_type='model').private
print('Declared private artifact repository is accessible; credential is not displayed.')
def run_full(mode, directory, fit_directory=None):
    child_env = os.environ.copy()
    child_env['HF_TOKEN'] = hf_token
    command = [sys.executable, '-u', '-m', 'src.training.full_m1', '--config', config_path, '--output-dir', str(directory), '--mode', mode]
    if fit_directory is not None:
        command += ['--fit-dir', str(fit_directory)]
    try:
        return subprocess.run(command, env=child_env, check=False)
    finally:
        child_env.pop('HF_TOKEN', None)


Declared private artifact repository is accessible; credential is not displayed.


In [4]:
prep_dir = Path(tempfile.mkdtemp(prefix='full-m1-prepare-', dir=remote_root))
assert run_full('prepare', prep_dir).returncode == 0
plan = json.loads((prep_dir / 'plan.json').read_text())
masking = json.loads((prep_dir / 'masking_report.json').read_text())
assert plan['training_rows'] == 5040 and plan['development_rows'] == 840
assert plan['planned_update_attempts'] == 1200
assert masking['prefixes_verified'] and masking['truncated_examples'] == 0
assert all(group['rows'] == 2520 and group['max_tokens'] <= 256 for group in masking['by_mode'].values())
print(json.dumps(masking, indent=2))
print('Training F/R counts:', plan['training_rows_by_partition'])
print('Checkpoint attempts:', plan['checkpoint_attempts'])
print('Actual training will start fresh; no later-task or final-test training data.')


{
  "rows": 5040,
  "by_mode": {
    "answer_only": {
      "rows": 2520,
      "max_tokens": 107,
      "prompt_tokens": 224898,
      "supervised_response_tokens": 17964
    },
    "thinking": {
      "rows": 2520,
      "max_tokens": 141,
      "prompt_tokens": 214818,
      "supervised_response_tokens": 68217
    }
  },
  "prefixes_verified": true,
  "truncated_examples": 0,
  "supervision": "All assistant-response tokens and first EOS; thinking includes the constructed rationale."
}
{
  "by_mode": {
    "answer_only": {
      "max_tokens": 107,
      "prompt_tokens": 224898,
      "rows": 2520,
      "supervised_response_tokens": 17964
    },
    "thinking": {
      "max_tokens": 141,
      "prompt_tokens": 214818,
      "rows": 2520,
      "supervised_response_tokens": 68217
    }
  },
  "prefixes_verified": true,
  "rows": 5040,
  "supervision": "All assistant-response tokens and first EOS; thinking includes the constructed rationale.",
  "truncated_examples": 0
}
Training F/R c

In [5]:
import zipfile

fit_dir = Path(tempfile.mkdtemp(prefix='full-m1-fit-', dir=remote_root))
fit_result = run_full('fit', fit_dir)
fit_archive = Path(str(fit_dir) + '-evidence.zip')
with zipfile.ZipFile(fit_archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(fit_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
print('Disposable fit evidence:', fit_archive)
assert (fit_dir / 'fit_report.json').is_file(), 'Fit failed before a report; keep the ZIP and inspect the error.'
fit_report = json.loads((fit_dir / 'fit_report.json').read_text())
print(json.dumps(fit_report, indent=2))
assert fit_result.returncode == 0 and fit_report['success'], 'Stop after failed fit; preserve settings/evidence for review.'
assert fit_report['temporary_optimizer_updates'] == 1 and not fit_report['checkpoint_saved']
print('Probe process exited. Its temporary adapter is discarded; the next cell begins actual full training.')


Loading weights: 100%|██████████| 311/311 [00:01<00:00, 159.97it/s]


Disposable fit evidence: /kaggle/working/full-m1-fit-_lce78cr-evidence.zip
{
  "actual_training_starts_fresh_in_a_separate_process": true,
  "checkpoint_saved": false,
  "generation_max_new_tokens": 1024,
  "generation_peak_memory_gib": 5.435481548309326,
  "gpu": "Tesla T4",
  "gpu_total_memory_gib": 14.56219482421875,
  "limits": "Representative actual-data batch/generation probe, not a guarantee against worst-case full-run OOM. No learning metrics are reported.",
  "model_id": "Qwen/Qwen3-1.7B",
  "model_revision": "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e",
  "observed_generated_token_counts": [
    308,
    273,
    968,
    159,
    163,
    400,
    164,
    273,
    172,
    280,
    230,
    200,
    127,
    146,
    143,
    208
  ],
  "optimizer_state_populated": true,
  "probe_generation_batch_size": 16,
  "probe_max_training_tokens": 141,
  "probe_training_batch_size": 4,
  "purpose": "Disposable memory feasibility only; not an M1 result.",
  "seconds": 117.1324,
  "succe

In [6]:
# ACTUAL FULL TRAINING: a new process, fresh pinned base/new LoRA,1200 attempts.
assert fit_report['success']
output_dir = Path(tempfile.mkdtemp(prefix='m1-full-qwen17-v1-', dir=remote_root))
print('Full training output:', output_dir, flush=True)
print('Private checkpoints back up every300 attempts; only the1200 endpoint can qualify.', flush=True)
training_result = run_full('train', output_dir, fit_dir)
print('Training process return code:', training_result.returncode)
if training_result.returncode == 2:
    print('Baseline replay/utility guard failed before training. Preserve the evidence; do not retry selectively.')
elif training_result.returncode != 0:
    print('Run incomplete. Keep checkpoint receipts/partial logs and continue to the backup/ZIP cells for evidence.')


Full training output: /kaggle/working/m1-full-qwen17-v1-hpcajqfx
Private checkpoints back up every300 attempts; only the1200 endpoint can qualify.


Loading weights: 100%|██████████| 311/311 [00:01<00:00, 166.24it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


base_controls_answer_only.jsonl: 16/16
base_controls_thinking.jsonl: 16/16
Thinking=True: generated 10/10 predictions
Mixed training 10/1200; response CE=2.0377
Mixed training 20/1200; response CE=0.9693
Mixed training 30/1200; response CE=0.9922
Mixed training 40/1200; response CE=0.7747
Mixed training 50/1200; response CE=0.6581
Mixed training 60/1200; response CE=0.5873
Mixed training 70/1200; response CE=0.8518
Mixed training 80/1200; response CE=0.9059
Mixed training 90/1200; response CE=0.7502
Mixed training 100/1200; response CE=0.6472
Mixed training 110/1200; response CE=0.5327
Mixed training 120/1200; response CE=0.8546
Mixed training 130/1200; response CE=0.5809
Mixed training 140/1200; response CE=0.8306
Mixed training 150/1200; response CE=0.6445
Mixed training 160/1200; response CE=0.5578
Mixed training 170/1200; response CE=0.6320
Mixed training 180/1200; response CE=0.6983
Mixed training 190/1200; response CE=0.7525
Mixed training 200/1200; response CE=0.4234
Mixed train

Loading weights: 100%|██████████| 311/311 [00:01<00:00, 167.89it/s]


reloaded_dev_answer_only.jsonl: 16/840
reloaded_dev_answer_only.jsonl: 32/840
reloaded_dev_answer_only.jsonl: 48/840
reloaded_dev_answer_only.jsonl: 64/840
reloaded_dev_answer_only.jsonl: 80/840
reloaded_dev_answer_only.jsonl: 96/840
reloaded_dev_answer_only.jsonl: 112/840
reloaded_dev_answer_only.jsonl: 128/840
reloaded_dev_answer_only.jsonl: 144/840
reloaded_dev_answer_only.jsonl: 160/840
reloaded_dev_answer_only.jsonl: 176/840
reloaded_dev_answer_only.jsonl: 192/840
reloaded_dev_answer_only.jsonl: 208/840
reloaded_dev_answer_only.jsonl: 224/840
reloaded_dev_answer_only.jsonl: 240/840
reloaded_dev_answer_only.jsonl: 256/840
reloaded_dev_answer_only.jsonl: 272/840
reloaded_dev_answer_only.jsonl: 288/840
reloaded_dev_answer_only.jsonl: 304/840
reloaded_dev_answer_only.jsonl: 320/840
reloaded_dev_answer_only.jsonl: 336/840
reloaded_dev_answer_only.jsonl: 352/840
reloaded_dev_answer_only.jsonl: 368/840
reloaded_dev_answer_only.jsonl: 384/840
reloaded_dev_answer_only.jsonl: 400/840
reload

In [7]:
summary_path = output_dir / 'summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print('Execution complete:', summary['execution_complete'])
    print('Applied updates:', summary.get('optimizer_updates', 0))
    if 'm1_candidate' in summary:
        print('Full-M1 candidate gates:', json.dumps(summary['m1_candidate'], indent=2))
        for stage, groups in summary['evaluation'].items():
            for suite, result in groups.items():
                print(stage, suite, '| matches:', str(result['answer_matches']) + '/' + str(result['count']), '| format:', result['format_violations'], '| caps:', result['token_limit_hits'])
    else:
        print('Stop reason:', summary.get('stop_reason'))
else:
    print('No completed summary. Preserve partial checkpoints and failure evidence.')
print('Gate completion is distinct from independent review/canonical M1 promotion; this run does no unlearning.')


Execution complete: True
Applied updates: 1200
Full-M1 candidate gates: {
  "by_fact_mode": {
    "dev_answer_only": {
      "base_utility_accuracy": 1.0,
      "checks": {
        "F/author_book": true,
        "F/multi_hop": true,
        "R/author_book": true,
        "R/multi_hop": true,
        "adapter_changed": true,
        "format": true,
        "generation_limit": true,
        "optimizer_budget": true,
        "save_reload": true,
        "utility_baseline_measurable": true,
        "utility_preserved": true
      },
      "dev": {
        "F/author_book": {
          "accuracy": 0.9833333333333333,
          "correct": 59,
          "count": 60
        },
        "F/multi_hop": {
          "accuracy": 0.9,
          "correct": 81,
          "count": 90
        },
        "R/author_book": {
          "accuracy": 0.9944444444444445,
          "correct": 179,
          "count": 180
        },
        "R/multi_hop": {
          "accuracy": 0.8925925925925926,
          "correc

In [8]:
# Preserve the endpoint and all evaluation evidence privately, including failed-gate runs.
from src.training.full_artifacts import publish_verified
from src.m0_smoke_test import write_json

remote_path = config['persistence']['path_prefix'] + '/' + output_dir.name
try:
    if (output_dir / 'artifact_manifest.json').is_file():
        receipt = publish_verified(output_dir, config['persistence'], remote_path, hf_token)
        print('Endpoint adapter/optimizer/tokenizer immutable round-trip verified.')
        print('Private revision:', receipt['artifact_revision'])
    else:
        assert api.repo_info(config['persistence']['repo_id'], repo_type='model').private
        preserved = api.upload_folder(repo_id=config['persistence']['repo_id'], repo_type='model', folder_path=str(output_dir), path_in_repo=remote_path, commit_message='Preserve incomplete full-M1 attempt evidence')
        write_json(output_dir / 'partial_persistence.json', {'repo_id': config['persistence']['repo_id'], 'path': remote_path, 'artifact_revision': preserved.oid, 'execution_complete': False, 'canonical_m1': False, 'note': 'Partial evidence upload only; no endpoint round-trip claimed.'})
        print('Partial evidence backed up. No completed endpoint artifact is claimed.')
except Exception as backup_error:
    write_json(output_dir / 'persistence_failure.json', {'error_type': type(backup_error).__name__, 'message': str(backup_error).replace(hf_token, '[redacted]'), 'verified': False})
    print('Private backup failed:', type(backup_error).__name__, '| retain this runtime/output and download evidence for review.')


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

shared/m1-full-qwen17-v1/m1-full-qwen17-(…): reconstructing file:   0%|          |  0.00B / 34.9MB            

shared/m1-full-qwen17-v1/m1-full-qwen17-(…): downloading bytes:           |  0.00B            

shared/m1-full-qwen17-v1/m1-full-qwen17-(…): reconstructing file:   0%|          |  0.00B / 70.1MB            

shared/m1-full-qwen17-v1/m1-full-qwen17-(…): downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/4.17k [00:00<?, ?B/s]

shared/m1-full-qwen17-v1/m1-full-qwen17-(…): reconstructing file:   0%|          |  0.00B / 11.4MB            

shared/m1-full-qwen17-v1/m1-full-qwen17-(…): downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/720 [00:00<?, ?B/s]

Endpoint adapter/optimizer/tokenizer immutable round-trip verified.
Private revision: a98bc31bcb3f7c404ea91ead35689301ab0c40d3


In [9]:
archive = Path(str(output_dir) + '-evidence.zip')
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_DEFLATED) as bundle:
    for file in sorted(output_dir.iterdir()):
        if file.is_file() and file.suffix in {'.json', '.jsonl', '.txt'}:
            bundle.write(file, arcname=file.name)
    for checkpoint in sorted((output_dir / 'checkpoints').glob('attempt-*')):
        for name in ('checkpoint.json', 'artifact_manifest.json', 'persistence.json'):
            file = checkpoint / name
            if file.is_file():
                bundle.write(file, arcname=str(file.relative_to(output_dir)))
print('Download from Kaggle Output:', archive)
print('Adapter/optimizer binaries remain in private HF storage; this ZIP contains text evidence only.')
print('Endpoint backup receipt present:', (output_dir / 'persistence.json').is_file())


Download from Kaggle Output: /kaggle/working/m1-full-qwen17-v1-hpcajqfx-evidence.zip
Adapter/optimizer binaries remain in private HF storage; this ZIP contains text evidence only.
Endpoint backup receipt present: True
